In [1]:
!apt-get -qq update
!apt-get -qq install openjdk-17-jdk-headless
!pip -q install pyspark==3.5.1 pyarrow

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 1.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.0.1 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.1 which is incompatible.


In [2]:
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("CSCI316_Taxi_Project1")
    .master("local[*]")
    .config("spark.driver.memory", "6g")
    .config("spark.sql.shuffle.partitions", "200")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("Spark version:", spark.version)


Spark version: 3.5.1


In [3]:
data_paths=[
    "/content/yellow_tripdata_2025-01.parquet",
    "/content/yellow_tripdata_2025-02.parquet",
    "/content/yellow_tripdata_2025-03.parquet"
]

In [4]:
df1 = spark.read.parquet(data_paths[0])
df1.printSchema()
df1.show(5, truncate=False)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

+--------+--------------------+---------------------+---------------+------

In [5]:
df = spark.read.parquet(*data_paths)

df.printSchema()
df.select("tpep_pickup_datetime", "tpep_dropoff_datetime", "PULocationID", "DOLocationID", "trip_distance", "passenger_count").show(5, truncate=False)

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)

+--------------------+---------------------+------------+------------+-----

In [6]:
from pyspark.sql import functions as F

df2 = (
    df.dropna(subset=[
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_distance",
        "passenger_count"
    ])
    .withColumn(
        "trip_duration_min",
        (F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) / 60.0
    )
    .filter(F.col("trip_duration_min").between(1, 180))
    .filter(F.col("trip_distance").between(0.1, 100))
    .filter(F.col("passenger_count").between(1, 6))
)

df2.select("tpep_pickup_datetime","tpep_dropoff_datetime","trip_distance","passenger_count","trip_duration_min").show(5, truncate=False)

+--------------------+---------------------+-------------+---------------+------------------+
|tpep_pickup_datetime|tpep_dropoff_datetime|trip_distance|passenger_count|trip_duration_min |
+--------------------+---------------------+-------------+---------------+------------------+
|2025-03-01 00:17:16 |2025-03-01 00:25:52  |0.9          |1              |8.6               |
|2025-03-01 00:37:38 |2025-03-01 00:43:51  |0.6          |1              |6.216666666666667 |
|2025-03-01 00:24:35 |2025-03-01 00:39:49  |1.94         |1              |15.233333333333333|
|2025-03-01 00:56:16 |2025-03-01 01:01:35  |0.95         |2              |5.316666666666666 |
|2025-03-01 00:01:44 |2025-03-01 00:10:00  |1.5          |1              |8.266666666666667 |
+--------------------+---------------------+-------------+---------------+------------------+
only showing top 5 rows



In [7]:
df3 = (
    df2
    .withColumn("pickup_hour", F.hour("tpep_pickup_datetime"))
    .withColumn("pickup_dayofweek", F.dayofweek("tpep_pickup_datetime"))
    .withColumn("pickup_month", F.month("tpep_pickup_datetime"))
    .withColumn("is_weekend", F.when(F.col("pickup_dayofweek").isin([1,7]), 1).otherwise(0))
)

In [8]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler

feature_cols_num = [
    "trip_distance",
    "passenger_count",
    "pickup_hour",
    "pickup_dayofweek",
    "pickup_month",
    "is_weekend"
]

feature_cols_cat = [
    "PULocationID",
    "DOLocationID",
    "payment_type"
]

existing_cols = set(df3.columns)
feature_cols_cat = [c for c in feature_cols_cat if c in existing_cols]

indexers = [
    StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
    for c in feature_cols_cat
]

encoders = [
    OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_ohe")
    for c in feature_cols_cat
]

assembled_inputs = feature_cols_num + [f"{c}_ohe" for c in feature_cols_cat]
assembler = VectorAssembler(inputCols=assembled_inputs, outputCol="features_raw")

scaler = StandardScaler(inputCol="features_raw", outputCol="features", withMean=True, withStd=True)

pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler])

prep_model = pipeline.fit(df3)
df_ready = prep_model.transform(df3).select(
    "features",
    F.col("trip_duration_min").alias("label")
)

df_ready.show(3, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

In [9]:
df_trainable = df_ready.sample(False, 0.02, seed=42)
train_df, test_df = df_trainable.randomSplit([0.8, 0.2], seed=42)

print("Train approx:", train_df.count())
print("Test approx:", test_df.count())

Train approx: 139269
Test approx: 34780


In [10]:
from pyspark.ml.regression import LinearRegression

lr = LinearRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=50,
    regParam=0.05,
    elasticNetParam=0.0
)

lr_model = lr.fit(train_df)
lr_preds = lr_model.transform(test_df)

lr_preds.select("label", "prediction").show(10, truncate=False)


+------------------+------------------+
|label             |prediction        |
+------------------+------------------+
|1.4666666666666666|11.058375698747113|
|1.6333333333333333|9.934060031431727 |
|3.8666666666666667|10.407420468633063|
|1.95              |9.009774890961125 |
|79.41666666666667 |10.063659249048214|
|2.966666666666667 |4.063668664457509 |
|2.1333333333333333|10.96116662552231 |
|1.8833333333333333|10.721544461906888|
|2.316666666666667 |6.425298792193644 |
|1.3333333333333333|7.834737833181862 |
+------------------+------------------+
only showing top 10 rows



In [11]:
from pyspark.ml.evaluation import RegressionEvaluator

def eval_regression(pred_df, label_col="label", pred_col="prediction"):
    metrics = {}
    for metric in ["rmse", "mae", "r2"]:
        evaluator = RegressionEvaluator(
            labelCol=label_col,
            predictionCol=pred_col,
            metricName=metric
        )
        metrics[metric] = evaluator.evaluate(pred_df)
    return metrics

lr_metrics = eval_regression(lr_preds)
print("Linear Regression metrics:", lr_metrics)


Linear Regression metrics: {'rmse': 2542748.342896256, 'mae': 17470.1119488199, 'r2': -42646092103.931274}


In [12]:
from pyspark.ml.regression import DecisionTreeRegressor

dt = DecisionTreeRegressor(
    featuresCol="features",
    labelCol="label",
    maxDepth=10,
    minInstancesPerNode=50
)

dt_model = dt.fit(train_df)
dt_preds = dt_model.transform(test_df)

dt_metrics = eval_regression(dt_preds)
print("Decision Tree metrics:", dt_metrics)


Decision Tree metrics: {'rmse': 5.789592221290967, 'mae': 3.5386477601612207, 'r2': 0.778910339467016}


In [13]:
from pyspark.sql.functions import rand, floor

df_cv = df_ready.withColumn(
    "fold_id",
    floor(rand(seed=42) * 10)
)

df_cv.select("fold_id").groupBy("fold_id").count().show()


+-------+------+
|fold_id| count|
+-------+------+
|      0|866574|
|      7|866095|
|      6|866338|
|      9|867900|
|      5|868732|
|      1|870290|
|      3|866718|
|      8|868136|
|      2|867040|
|      4|867737|
+-------+------+



In [ ]:
from pyspark.ml.regression import DecisionTreeRegressor

cv_results = []

for fold in range(10):
    print(f"Running fold {fold}...")

    train_fold = df_cv.filter(df_cv.fold_id != fold)
    val_fold   = df_cv.filter(df_cv.fold_id == fold)

    dt = DecisionTreeRegressor(
        featuresCol="features",
        labelCol="label",
        maxDepth=10,
        minInstancesPerNode=50
    )

    model = dt.fit(train_fold)
    preds = model.transform(val_fold)

    metrics = eval_regression(preds)
    metrics["fold"] = fold

    cv_results.append(metrics)


Running fold 0...


In [ ]:
import pandas as pd

cv_df = pd.DataFrame(cv_results)
cv_df

In [ ]:
cv_df.mean()

In [ ]:
cv_df.std()